# Атака RSA Blinding
## Введение
RSA (названа в честь своих создателей Ronald Linn Rivest, Adi Shamir and Leonard Adleman) - это пример асимметричной криптосистемы, которая может быть использована для безопасной передачи данных и создания подписей. Хоть от неё постепенно избавляются, её всё ещё можно встретить во многих продуктах и системах, поэтому есть смысл понимать, как она работает. Начнем с основ

## Группы и Поля
Группа - это множество $\mathbb{G}$ определенной на нем операцией с двумя аргументами $*, \forall a,b \in \mathbb{G}\ \Rightarrow a*b \in \mathbb{G}$ и свойствами:

+ Ассоциативности ($\forall a,b,c\in \mathbb{G},\ (a*b)*c = a*(b*c)$)

+ Наличия единичного элемента ($\exists\ e \in \mathbb{G}:\forall a \in \mathbb{G}, \ a*e = e*a = a$)

+ Наличия обратного элемента ($\forall a\in \mathbb{G},\ \exists b \in \mathbb{G}: a*b=e=b*a$)

Группа называется коммутативной (абелевой), если она удовлетворяет свойству коммутативности:

+ $\forall a,b \in \mathbb{G},\ a*b=b*a$

Т.е., если у нас есть некоторое множество, результат применения оператора к любым двум элементам также содержится в этом множестве, порядок вычисления выражений не важен, есть некий элемент, который не изменяет другие элементы (1) и для любого элемента всегда есть такой парный, что при применении к ним оператора получается единичный (1), то у нас есть группа. Если ещё и можно поменять местами аргументы при вызове оператора и ничего не изменится, то аж абелева.

Если мы используем аддитивную нотацию (ставим плюс), то группа называется аддитивной. Если мультипликативную (ставим знак умножения) - мультипликативной.

Давайте рассмотрим простой пример аддитивной группы целых чисел по модулю некоторого числа $n$. Например, если $n=5$, то группа содержит элементы $\{0,1,2,3,4\}$. Если происходит переполнение (результат меньше $0$ или больше или равен $n$), мы тут же добавляем или вычитаем $k*n, k \in \mathbb{N}$ из результата, чтобы он снова был в множестве. Как видно, $0$ - единичный элемент, $1$ - обратный элемент к $4$, а $2$ - к $3$. Ассоциативность очевидна, а поскольку мы легко можем менять элементы местами при сложении, то эта группа ещё и абелева.

О полях можно думать как о группах с двумя операциями (это не совсем верно, но так проще вникнуть). Допустим у Вас есть множество с двумя операциями $(+,*)$. Оно будет полем, если:

1. Для обоих операций оно является абелевой группой, за исключением единичного элемента аддитивной группы, который не входит в мультипликативную (у $0$ же не может быть обратного элемента).

2. Действует закон дистрибутивности: $a*(b+c)= a*b+a*c$.

В качестве примера давайте посмотрим на поле $F_n$, где $n=5$:

1. Сложение осталось таким же, как и в прошлом примере

2. Таблица умножения: 


|     | 0   | 1   | 2   | 3   | 4   |
| --- | --- | --- | --- | --- | --- |
| 0   | 0   | 0   | 0   | 0   | 0   |
| 1   | 0   | 1   | 2   | 3   | 4   |
| 2   | 0   | 2   | 4   | 1   | 3   |
| 3   | 0   | 3   | 1   | 4   | 2   |
| 4   | 0   | 4   | 3   | 2   | 1   |

Можно видеть, что в каждом ненулевом ряду есть по $1$, так что у каждого элемента есть обратный. Если удалить ряд и столбец, содержащие нулевой элемент, то как раз получится таблица мультипликативной группы по модулю $n$ или $\mathbb{Z}_n^{*}$, которая содержит элементы $\{1,2,3,4\}$. Здесь же можно заметить интересную особенность групп. Если порядок (количество элементов) в группе не простое, то можно генерировать подгруппы. Это группы, которые используют ту же операцию, что и оригинальная (например, умножение по модулю $5$), но состоят из подмножества элементов оригинальной группы. В данном случае элементы $\{1,4\}$ образуют такую подгруппу, т.к.  $4\cdot 4 = 1\ mod\ 5$ (получаем замкнутое по умножению множество). Подгруппы упрощают вычисление дискретного логарифма (но об это позже). Из-за этого в криптографии часто используются большие безопасные простые числа $p=2*q+1$, где $q$ - это другое простое число. Таким образом получается всего 2 нетривиальных подгруппы с порядками $2$ и $q$.

## RSA
RSA использует мультипликативную группу по модулю $N=pq$, где $p$ и $q$ - простые числа. Степень мультипликативной группы (по-сути, её мощность) может быть вычислена при помощи функции Эйлера  для составного числа из двух простых: $\varphi(N)=(p-1)(q-1)$. Функция считает количество натуральных чисел меньше $N$, которые не кратны $p$ или $q$. 

Например, если мы возьмем $N=13\cdot 17=221$, то элемент $26$ не состоит в мультипликативной группе, т.к. $26\cdot 17 = 0\ \mathit{mod}\ 221$, а $0$ не в группе. Т.е. надо исключить все элементы кратные $p$ (таких $q$) и $q$ (таких $p$). Т.е. всего элементов в группе будет $p\cdot q - p - q +1=(p-1)\cdot(q-1)$ (добавляем $1$, потому что $0$ посчитали до этого два раза).

Поскольку все остальные числа взаимно просты с $N$, они состоят в мультипликативной группе. Как мы знаем, если возвести любой элемент конечной мультипликативной группы в степень этой группы, то получим нейтральный элемент (единицу): $a^{\varphi(N)}=1, a \in Z^{*}_{N}$. Поэтому в RSA используют два числа $e$ (открытая экспонента) и $d$ (закрытая экспонента), такие что $ed=1\ mod\ \varphi(N)$. Пара чисел $(e,N)$ используется как открытый ключ, а $(d,N)$ как закрытый. Вычисление $d$ из открытого ключа является сверхполиномиальной задачей (NP), если не были сгенерированы слабые $N$, $d$ или $e$. Одним из способов решения является факторизация $N$ в произведение $p$ и $q$.

Пусть дан открытый текст (число) $M, M < N$, открытый ключ $(e,N)$ и закрытый ключ $(d,N)$,  шифрование и расшифрование осуществляются следующим образом:

Шифрование
$C=M^{e}\ mod\ N$

Расшифрование

$M=C^{d} \ mod\ N$

Проверка корректности:

$C^{d}\ mod\ N=M^{ed}\ mod\ N= M^{ed\ \mathit{mod}\ \varphi(N)}\ mod\ N=M^{1}\ mod\ N= M\ mod\ N$

## Подготовка
Попробуем немного поработать с RSA. Если ещё не установили, установите Pycryptodome. На Linux и Windows должна сработать следующая команда (Предварительно надо установить python 3 и pip, но я надеюсь, что вы справились с этим самостоятельно):

In [1]:
!python3 -m pip install --user pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 11.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


После установки надо перезапустить ядро jupyter (круговая стрелка рядом с "Run"). Если возникнут проблемы, загляните в документацию: [Pycryptodome installation](https://pycryptodome.readthedocs.io/en/latest/src/installation.html).

## Примитивная реализация RSA
Давайте сделаем простейшую версию RSA. Будем использовать открытую экспоненту $e=65537$. Обычно используют эту константу, потому что она переполняет модуль даже при открытом тексте $M=2$ и у нее удобное двоичное представление $65537_{10}=10000000000000001_{2}$, которое позволяет эффективно возводить число в степень, используя алгоритм "Square and multiply".

Сначала сгенерируем $p$ и $q$. Функция getStrongPrime дает возможность выбрать количество бит в генерируемом простом числе и проверяет, что $НОД(p-1,e)=1$

In [2]:
try:
    from Crypto.Util.number import getStrongPrime, inverse,bytes_to_long, long_to_bytes
except ImportError:
    print ("Pycryptodome not installed")

In [13]:
e=65537
p=getStrongPrime(1024,e=e)
q=getStrongPrime(1024,e=e)

assert p.bit_length() == 1024
assert q.bit_length() == 1024

print(f"p: {p}")

p: 145717331131257078280574969110513933027456769286237736207463031266190855742416937693746217623173278898460511557197967966424351164756948410232382590360598851432900265521897817201791915026989283526413222625476113890796035940314645370083997061181251497654313853714022427492435897898905918960069984132823286300181


In [6]:
N=p*q
phi=(p-1)*(q-1)
d=inverse(e,phi)
public_key=(e,N)
private_key=(d,N)

print(f"pk: {public_key}")
print(f"sk: {private_key}")

pk: (65537, 26951053654104985609833619424625180918738597662816998225118725789492466984194747108392842617258619105733881128825950199102627895798416570299036995015262367534740821962434146039416232717681573751664876301688351934148195553025724812820647568841669033438489796697390576290940470032429601175408639204020732478949021965030602463386643972430870950931635850550707787443041623344641683095225981870683337532441151366848247152690584770466245468182809031697954247547735110141870295054391898983493287485647554647244409364706511790608545138344065119031176830347314751509952982032150060521965912102182763196317209798239278946355889)
sk: (1676930689392393767181584956432802275820258076345196688319409494245882201934011928507626739470180157611145161773142800584413324282264722070973847723777970952792561816964675842951790801778566633268825740925351503991173105667459612764088021365374643095586824129463040603005890545771723265835808000765304223608907683756227374648182674331916178359231814628194906916699

Мы успешно сгенерировали ключи, теперь давайте зашифруем сообщение, расшифруем закрытый текст и проверим, что получили то же самое

In [18]:
M=bytes_to_long(b'Hello, RSA!')
print(f"Message: {M}\nType: {type(M)}\n")

# Exponentiating (Encrypt)
C=pow(M,e,N)


print(f'CipherText (Hex): {hex(C)}\nType: {type(C)}\n')

# Exponentiating (Decrypt)
M1=pow(C,d,N)

assert M1==M
print ('M1:',long_to_bytes(M1))

Message: 87521618088895491219865889
Type: <class 'int'>

CipherText (Hex): 0x50467d79b2048b0340029f3b31ad20b383cb6c62b48adc9d1f2bbe2640a22b22491c55da694f838c4cb5badc170807f08d2e7f2f62f899b0297bf8daf4b137c82e9749321a4577bc7497399032b48f1ae7bda79b27fa181138bfaa73be5e441d58a7e4331acada54cde364c385f9d627627928733f7783e9d1ed4656188a8664431fc17ff7c98b40ce292373899559c11ee3c01a8d39e5f468a94aae6f6b5aa0364448d5170221f2b498c44e5b04bd3c11ca0bf4d3794462b25773f9e4a8ae9143f70e8ce4495df2e8e17712a0e978ec248ca257da16fc35da4b5d4648ebed12f178573a2abe0d8de7d1d4feb94b7a56916aa8dcc781e51014d74fa456a289dc
Type: <class 'int'>

M1: b'Hello, RSA!'


Создание подписи - обратная операция к шифрованию. 
$$Sign(M)\equiv Dec(M),\space Check(S) \equiv Enc(S)$$
Таким образом любой, владеющий открытым ключом, может проверить правильность подписи, а создать её может только сторона, у которой есть закрытый ключ.
Поздравляю, теперь вы знаете, как шифровать и создавать подписи при помощи RSA. Дальше рассмотрим одно из его интересных свойств.

## RSA Blinding
RSA - это гомоморфное шифрование по отношению к операции умножения.
Отображение является гомоморфизмом групп, если оно сохраняет отношения между элементами. Если ничего не понятно, не беспокойтесь, я в первый раз, когда услышал, тоже ничего не понял. Что это значит на практике: пусть у вас есть два элемента группы $G_1$ $(x,y)$ и вы применяете к ним гомоморфное отображение, они будут также связаны в новой группе $G_2$ (для RSA $G_1= G_2$): 
$$\varphi(x\cdot y)=\varphi(x)\times\varphi(y)$$
Для шифрования RSA: $$Enc(M_1 \cdot M_2)=Enc(M_1)\times Enc(M_2)$$
То же самое верно и для расшифрования:
$$Dec(C_1 \times C_2)=Dec(C_1) \cdot Dec(C_2)$$
Протестируем это свойство в python

In [23]:
class BasicRSA:
    def __init__(self, e,p,q):
        self.e=e
        self.p=p
        self.q=q
        self.N=p*q
        self.d=inverse(e, (p-1)*(q-1))
    
    def encryptNumber(self, m):
        return pow(m, self.e, self.N)
    
    def decryptNumber(self, c):
        return pow(c, self.d, self.N)

base_rsa = BasicRSA(e,p,q) #we created these parameters earlier

m1 = 2 # Message 1
m2 = 3 # Message 2
m3 = m1*m2 # Message 3

c1 = base_rsa.encryptNumber(m1) # CipherText1
c2 = base_rsa.encryptNumber(m2) # CipherText2

print('c1:',c1)
print('c2:',c2)

c3 = (c1*c2) % base_rsa.N # CipherText 3
print('c3:',c3)

m3_dec = base_rsa.decryptNumber(c3)
print ('m3: %d, m3_dec: %d'%(m3,m3_dec))

assert m3_dec==m3

c1: 8112479406250531076126614788711293935308971373199960936584714447315613806335940950118461709556109935287197596639215206250034372004057189993655014234362093961868109419002086711222304518742921802378734909505445351887334514098214483797622822466974013470462156624591788829563673516646433191807765409504292233333027114529747200941473602347199073253273566789377302900599488733188338814573595514264741256396738527585958957366979720830932205962009415715877910912787602959575025579995617875539547226352554145889322719992354766194032934754503720688862069951207023008023318052056099903756148219930624584155852994834490693927825
c2: 499268222670506116930137528471644624985016717241259499485296799426430574512449143006603982775470156301822813092775379308945900814909485559521033561880504349358493930198182655995348739528425786245380850259724090465129427048610552605440624144596302751561764255367105249635419458669990474996282100455189045555958636727146676234317723075335752982261981876378497318564309026308490

## Атакуем сервер
Теперь попробуйте применить эти знания к уязвимому серверу. Вы можете приконнеrтиться, используя ```nc cryptotraining.zone 1337``` или при помощи питоновских сокетов.

In [24]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Ининциализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1337))
        if show:
            print (self.recv_until().decode())
    def recv_until(self,symb=b'\n>'):
        """Получение сообщения с сервера, по умолчанию до приглашения к вводу команды"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def get_public_key(self,show=True):
        """Получение открытого ключа с сервера"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        e=int(re.search(r'(?<=e: )\d+',response).group(0))
        N=int(re.search(r'(?<=N: )\d+',response).group(0))
        self.num_len=len(long_to_bytes(N))
        return (e,N)
    
    def signBytes(self,m,show=True):
        """Получение подписи для выбранного сообщения в байтах с сервера"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(m)>num_len:
            print ("The message is too long")
            return
        if len(m)<num_len:
            m=bytes((num_len-len(m))*[0x0]) +m
        hex_m=m.hex().encode()
        self.s.sendall(b'sign '+hex_m+b'\n')
        response=self.recv_until().decode()
        if show:
            print (response)
        if response.find('flag')!=-1:
            print('You tried to submit \'flag\'')
            return None
        signature_hex=re.search(r'(?<=Signature: )[0-9a-f]+',response).group(0)
        signature_bytes=bytes.fromhex(signature_hex)
        return bytes_to_long(signature_bytes)
    
    
    def signNumber(self,m,show=True):
        """Получение подписи с сервера для выбранного сообщения в числовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        return self.signBytes(long_to_bytes(m,num_len),show)
        
    def checkSignatureNumber(self,c,show=True):
        """Проверка сигнатуры (на сервере) для подписи в числовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        signature_bytes=long_to_bytes(c,num_len)
        self.checkSignatureBytes(signature_bytes,show)
    
    def checkSignatureBytes(self,c,show=True):
        """Проверка сигнатуры (на сервере) для подписи в байтовом представлении"""
        try:
            num_len=self.num_len
        except AttributeError:
            print ('You need to get the public key from the server first')
            return
        if len(c)>num_len:
            print ("The message is too long")
            return
        
        hex_c=c.hex().encode()
        self.s.sendall(b'flag '+hex_c+b'\n',)
        response=self.recv_until(b'\n').decode()
        
        if show:
            print (response)
        
        if response.find('Wrong')!=-1:
            print('Wrong signature')
            x=self.recv_until()
            if show:
                print (x)
            return
        flag=re.search(r'CRYPTOTRAINING\{.*\}',response).group(0)
        print ('FLAG: ',flag)
        
    def __del__(self):
        self.s.close()

In [40]:
vs=VulnServerClient()
(e,N)=vs.get_public_key()

Welcome to RSA blinding task
Available commands:
help - print this help
public - show public key
sign <hex(data)> - sign data
flag <hex(signature(b'flag'))> - print flag 
quit - quit
>
e: 65537
N: 20159717663186764200842482638329142432479376755681286432561400011207751568770239378735042390550988864636478212097889382541806378632813451522011734778394352464750695430236459156439656932108536936107092785759187120915559173321302027525229018106368725032056109022369913503577023942696069608771010384365856481001383579432844112231215767630328627015097422540087789462404508697086321213990868031273219614897901436844999442259387453021270642395531884848697650933478124254071912232445708062597679170291021925633789812405697682134528381868778865376836541179591638312152472136313757252384761293684336082840137773984575947459061
>


Вы можете подписывать сообщения при помощи методов signNumber (подписать число) и signBytes (подписать сообщение из байтов)

Проверять подпись можете при помощи методов checkSignatureNumber и checkSignatureBytes.

Ваша цель - получить правильную подпись для сообщения 'flag'.

Помните, что RSA - это гомоморфизм и решите задание.

Удачи!

In [41]:
vs.signBytes(b'flag')

I found 'flag' in your message for signing. Despicable...
>
You tried to submit 'flag'


Используем свойство гомоморфизма RSA:

$RSA_{Sign}(msg \cdot r \mod N) = RSA_{Sign}(msg) \cdot RSA_{Sign}(r) \mod N$

Значит, если сервер блокирует ```flag``` по байтам, то нужно добавить ```r,``` то есть случайность, которую мы знаем.

Сервер подпишет сообщение и зная $r^{-1}$, мы сможем получить корректную подпись и для ```msg.```

In [49]:
import random

msg = b'flag'
m = bytes_to_long(msg)

r = random.randrange(2, N)
r_inv = inverse(r, N)

blinded = (m * pow(r, e, N)) % N
blinded_bytes = long_to_bytes(blinded, vs.num_len)

# Signature
s_prod = vs.signBytes(blinded_bytes)

Signature: 08e059fb4a035aab9c9da410ea36ba3baf0c24517260fb6b4a622aba17c0bd58dd981c818ed421836a1eb71d63d32672ecaf66dff874f69dc9f27c691770ed6504cd64f9418fc7ee419885b4a73d86877e7106071136fb03803a5306ee19ad959a135f35fce5c46a306fa59b413b5e0c341fa6dd5bec9944ac30608b58fca2794654265077dd3b06fae2821d07c42467d7e8d9578827110a2eaccba1e42debb530309d1451255d7a3eac3641c72cc3eecd018fea8ba984a4efc9f9005f14e244070d428a8625fd5b9c0cfdb90373756a039bcc558ec3ec250c12e3c55adc6be9d6170ceb36ddbfbed8d0e073be45e9ba714feb7847e5553bbae7609e5a7a1eec
>


Подпись произведения получена. Получаем для сообщения:

In [50]:
signature = (s_prod * r_inv) % N

Проверка

In [51]:
vs.checkSignatureNumber(signature, show=True)

Congratulations! Here is your flag: CRYPTOTRAINING{n0t_s0_bl1nd_4ft3r_4ll}

FLAG:  CRYPTOTRAINING{n0t_s0_bl1nd_4ft3r_4ll}
